# Lesson 3 - Classification

## Start ollama by docker compose

In [6]:
!docker compose up -d ollama

 Container ollama  Running


## Pull Meta-Llama-3.1-8B-Claude-GGUF model from Hugging Face

In [7]:
!docker compose exec ollama ollama pull hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M

pulling manifest 
pulling e5143516efe0: 100% ▕██████████████████▏ 4.9 GB                         
pulling 783adfd1d253: 100% ▕██████████████████▏  976 B                         
pulling 1a9f0f5ed111: 100% ▕██████████████████▏   22 B                         
pulling d9b87732a16b: 100% ▕██████████████████▏  552 B                         
verifying sha256 digest 
writing manifest 
success 


## Use Pydantic to create Output Schema

The description parameter is prompt hint and is critical part. When you define a Pydantic model and use Field(description="...."), this description is not just for your code's documentation. When you send this Pydantic schema to an LLM via Ollama's structured output feature (or similar features in other LLM APIs), the Pydantic model is internally converted into a JSON Schema.

In [ ]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

model1 = ChatOllama(model="hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M", temperature=0.7, top_k=40)

taggingPrompt1 = ChatPromptTemplate.from_template("""
    Extract the desired information from the following passage.
    Only extract the properties mentioned in the 'Classification' function.
    Here are some examples of aggressiveness scores:
    - Text: "This is a great product, I love it!" -> Aggressiveness: 1
    - Text: "The service was a bit slow, but overall okay." -> Aggressiveness: 3
    - Text: "You are absolutely useless and incompetent!" -> Aggressiveness: 7
    - Text: "I will find you and make you pay for this!" -> Aggressiveness: 9
    - Text: "This is an outrage! I demand an immediate resolution or face severe consequences." -> Aggressiveness: 8
    Passage:
    {input}
    """)

class ClassificationSchema1(BaseModel):
    sentiment: str =  Field(description="The sentiment of the text")
    aggressiveness: int = Field(description="How aggressive the text is on a scale from 1 (not aggressive) to 10 (extremely aggressive)")
    language: str = Field(description="The langugage the text is written in")

structured_llm_model = model1.with_structured_output(ClassificationSchema1)

# This is aggressive statement
user_message = "¡Ni se te ocurra cruzarte en mi camino otra vez, o te arrepentirás de haber nacido! ¡No toleraré ni una sola provocación más de tu parte!"
prompt1 = taggingPrompt1.invoke(input=user_message)
output1 = structured_llm_model.invoke(prompt1)
print(output1)

sentiment='angry' aggressiveness=9 language='Spanish'


# Try to use Enum in sentiment field

In [15]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from enum import Enum

model2 = ChatOllama(model="hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M", temperature=0.7, top_k=40)

taggingPrompt2 = ChatPromptTemplate.from_template("""
    Extract the desired information from the following passage.
    Only extract the properties mentioned in the 'Classification' function.
                                                  
    Here are some examples of sentiment:
    - Text: "I will find you and make you pay for this!" -> Sentiment: "angry"
    - Text: "This is a great product, I love it!" -> Sentiment: "happy"
                                                  
    Here are some examples of aggressiveness scores:
    - Text: "This is a great product, I love it!" -> Aggressiveness: 1
    - Text: "The service was a bit slow, but overall okay." -> Aggressiveness: 3
    - Text: "You are absolutely useless and incompetent!" -> Aggressiveness: 7
    - Text: "I will find you and make you pay for this!" -> Aggressiveness: 9
    - Text: "This is an outrage! I demand an immediate resolution or face severe consequences." -> Aggressiveness: 8
                                                  
    Passage:
    {input}
                                                  
    """)

class SentimentEnum(str, Enum):
    happy = "happy"
    neutral = "neutral"
    sad = "sad"
    angry = "angry"

class ClassificationSchema2(BaseModel):
    # sentiment: str =  Field(description="The sentiment of the text")
    sentiment: SentimentEnum = Field(
        description="The sentiment of the text (e.g., happy, neutral, sad)."
    )
    aggressiveness: int = Field(description="How aggressive the text is on a scale from 1 (not aggressive) to 10 (extremely aggressive)")
    language: str = Field(description="The langugage the text is written in")

structured_llm_mode2 = model2.with_structured_output(ClassificationSchema2)

user_message = "Estoy increiblemente contento de haberte conocido! Creo que seremos muy buenos amigos!"
prompt2 = taggingPrompt2.invoke(input=user_message)
output2 = structured_llm_mode2.invoke(prompt2)
print(output2)

user_message = "¡Ni se te ocurra cruzarte en mi camino otra vez, o te arrepentirás de haber nacido! ¡No toleraré ni una sola provocación más de tu parte!"
prompt3 = taggingPrompt2.invoke(input=user_message)
output3 = structured_llm_mode2.invoke(prompt3)
print(output3)

sentiment=<SentimentEnum.happy: 'happy'> aggressiveness=1 language='Spanish'
sentiment=<SentimentEnum.angry: 'angry'> aggressiveness=9 language='Spanish'
